# p17_jepa_basics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/javanmardi298/rag_ollama/blob/main/lessons/p17_jepa_basics/p17_jepa_basics.ipynb)

**روال هر گام:** ۱) متن را بخوان ۲) کد را ببین و کامل کن (✍️) ۳) سلول را اجرا کن (Shift+Enter) ۴) «بررسی خودکار» (✅) را اجرا کن ۵) امتحان کوتاه (📝). در پایان «گزارش نهایی درس» نتیجه‌ی همه‌ی گام‌ها را نشان می‌دهد.

همه‌ی کد همین‌جا نوشته و اجرا می‌شود؛ به فایل جداگانه‌ای نیاز نیست. سلول‌ها را **به ترتیب** اجرا کن.

In [ ]:
# 1) آماده‌سازی (Colab: پروژه را clone و نصب می‌کند؛ محلی: اگر داخل پوشه‌ی پروژه‌ای چیزی تغییر نمی‌کند)
import os, sys, subprocess
if 'google.colab' in sys.modules and not os.path.exists('repo'):
    subprocess.run(['git', 'clone', '--depth', '1', '-b', 'main', 'https://github.com/javanmardi298/rag_ollama.git', 'repo'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', '-r', 'repo/requirements.txt', '-e', 'repo'], check=True)
if os.path.exists('repo'):
    os.chdir('repo')
while not os.path.exists('lessons') and os.getcwd() != os.path.dirname(os.getcwd()):   # باز کردن از داخل lessons/...
    os.chdir(os.path.dirname(os.getcwd()))
print('پوشه‌ی کار:', os.getcwd())

### مدل زبانی درس‌ها
پیش‌فرض `offline` است (بدون نصب؛ شکل کار را نشان می‌دهد). برای جواب واقعی یکی از خط‌های زیر را از حالت توضیح (`#`) دربیاور. کلید API را **داخل notebook ننویس** (commit می‌شود)؛ از `.env` یا `getpass` استفاده کن.

In [ ]:
import os
os.environ['LEARN_PROVIDER'] = 'offline'      # 'offline' | 'ollama' | 'anthropic' | 'opencode' | 'openai_compatible'
# os.environ['LEARN_PROVIDER'] = 'ollama';     os.environ['LEARN_MODEL'] = 'qwen2.5:7b'
# os.environ['LEARN_PROVIDER'] = 'anthropic';  os.environ['LEARN_MODEL'] = 'claude-sonnet-5-5'; os.environ['LEARN_BUDGET_USD'] = '0.5'
#   import getpass; os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('ANTHROPIC_API_KEY: ')   # اشتراک Claude.ai کلید API نیست
# os.environ['LEARN_PROVIDER'] = 'opencode';   os.environ['LEARN_MODEL'] = 'anthropic/claude-sonnet-5-5'
# os.environ['LEARN_PROVIDER'] = 'openai_compatible'; os.environ['LEARN_API_BASE'] = 'http://localhost:1234/v1'; os.environ['LEARN_MODEL'] = '...'
from learn import llm as learn_llm
reply = learn_llm.get().chat([{'role': 'user', 'content': 'Reply with the single word OK.'}])
print(os.environ['LEARN_PROVIDER'], '->', reply.content[:80], f'({reply.latency_s:.1f}s)')

---

# گام ۱: patch و ماسک — داده‌ی ورودی JEPA

## هدف این گام
تصویر ۸×۸ ارقام دست‌نویس را به **patch** تقسیم می‌کنی و یک **ماسک بلوکی** می‌سازی که بخشی از patch ها را پنهان می‌کند. این همان آماده‌سازی داده در I-JEPA است.

## پیش‌نیاز
numpy پایه (reshape، transpose).

## مفهوم
Transformer تصویر را به صورت دنباله‌ای از **patch** می‌بیند. تصویر ۸×۸ را به شبکه‌ی ۴×۴ از patch های ۲×۲ تقسیم می‌کنیم: ۱۶ patch، هر کدام ۴ پیکسل.

I-JEPA در هر گام:
1. یک **بلوک** از patch ها را پنهان می‌کند (**هدف**، target). بلوک (مربعی مجاور) مهم است: اگر patch های پراکنده را پنهان کنی، مدل کافی است همسایه را کپی کند و چیز معناداری یاد نمی‌گیرد. بلوک مجاور مدل را مجبور می‌کند «ساختار» را بفهمد.
2. بقیه‌ی patch ها (**زمینه**، context) به encoder داده می‌شوند.
3. predictor، embedding هدف را از روی embedding زمینه حدس می‌زند.

در این گام فقط مرحله‌ی ۱ و آماده‌سازی داده:
- `patchify(img)` : آرایه‌ی `(8, 8)` ← `(16, 4)`؛ ترتیب patch ها سطر به سطر (patch شماره‌ی ۰ گوشه‌ی بالا-چپ، ۱ کنارش، …) و پیکسل‌های هر patch هم سطر به سطر.
- `block_mask(rng, block=2)` : آرایه‌ی بولی `(16,)` به نام **keep**: `True` یعنی patch دیده می‌شود. یک مربع `block×block` از شبکه‌ی ۴×۴ با گوشه‌ی بالا-چپ تصادفی (`rng.integers`) پنهان (`False`) می‌شود.

## تمرین کد

راهنمای `patchify`: `img.reshape(4, 2, 4, 2).transpose(0, 2, 1, 3).reshape(16, 4)`. دلیلش: reshape ابعاد را به (سطر patch، سطر درون patch، ستون patch، ستون درون patch) می‌شکند؛ transpose سطر-patch و ستون-patch را کنار هم می‌آورد.

### ✍️ کد تو
کد زیر را ببین، TODO ها را کامل کن و سلول را اجرا کن (Shift+Enter). این‌جا می‌نویسی، نه در فایل دیگر.

In [ ]:
"""Step 1: patches and block masks for a mini I-JEPA (8x8 digits -> 4x4 grid of 2x2 patches)."""
from __future__ import annotations

import numpy as np

GRID, PATCH = 4, 2


def patchify(img: np.ndarray) -> np.ndarray:
    """(8, 8) image -> (16, 4): patches row by row, pixels inside a patch row by row."""
    # TODO 1: reshape to (GRID, PATCH, GRID, PATCH), swap the two middle axes, flatten to (16, 4)
    raise NotImplementedError("TODO 1: patchify")


def block_mask(rng: np.random.Generator, block: int = 2) -> np.ndarray:
    """keep mask (16,) bool: True = patch is visible. One random block x block square of the 4x4 patch grid is hidden."""
    # TODO 2: keep = np.ones((GRID, GRID), bool); pick a top-left corner r, c with rng.integers(0, GRID - block + 1);
    #         set the block to False; return keep.reshape(-1)
    raise NotImplementedError("TODO 2: block_mask")

### ▶ اجرای نمایشی
اول کد را بخوان، بعد اجرا کن. این سلول از توابعی که بالا نوشتی استفاده می‌کند.

In [ ]:
from sklearn.datasets import load_digits
img = load_digits().images[3] / 16.0
print("\n".join("".join("#" if v > 0.4 else "." for v in row) for row in img))
keep = block_mask(np.random.default_rng(0)).reshape(GRID, GRID)
print("\nmask (X = hidden target):")
print("\n".join("".join("." if k else "X" for k in row) for row in keep))
print("patches:", patchify(img).shape)

### ✅ بررسی خودکار
تست‌ها را روی چیزی که در سلول‌های بالا نوشتی اجرا می‌کند. ✗ = هنوز نه؛ پیام را بخوان و سلول کد را اصلاح و دوباره اجرا کن.

In [ ]:
from learn.notebook import check_cell
check_cell('p17_jepa_basics', 'step01_patches_masks', globals())

<details><summary><b>🔍 این تست‌ها دقیقاً چه چیزی را می‌سنجند؟</b> (کد تست)</summary>

```python
import numpy as np

IMG = np.arange(64).reshape(8, 8)


def check_patchify_shape(m):
    """(8, 8) -> (16, 4)"""
    assert m.patchify(IMG).shape == (16, 4), m.patchify(IMG).shape


def check_first_patches(m):
    """patch 0 is the top-left 2x2, patch 1 its right neighbour"""
    p = m.patchify(IMG)
    assert p[0].tolist() == [0, 1, 8, 9], p[0].tolist()
    assert p[1].tolist() == [2, 3, 10, 11], p[1].tolist()
    assert p[4].tolist() == [16, 17, 24, 25], p[4].tolist()


def check_patchify_is_lossless(m):
    """no pixel is lost or duplicated"""
    assert sorted(m.patchify(IMG).reshape(-1).tolist()) == list(range(64))


def check_mask_hides_one_block(m):
    """exactly four patches are hidden"""
    keep = m.block_mask(np.random.default_rng(0))
    assert keep.dtype == bool and keep.shape == (16,), (keep.dtype, keep.shape)
    assert (~keep).sum() == 4, f"{(~keep).sum()} patches hidden"


def check_hidden_block_is_a_square(m):
    """the hidden patches form a 2x2 square in the grid"""
    for seed in range(20):
        hidden = np.argwhere(~m.block_mask(np.random.default_rng(seed)).reshape(4, 4))
        assert hidden[:, 0].max() - hidden[:, 0].min() == 1 and hidden[:, 1].max() - hidden[:, 1].min() == 1, f"seed {seed}: {hidden.tolist()}"


def check_mask_varies_and_block_size(m):
    """different seeds hide different blocks; block=3 hides nine"""
    masks = {m.block_mask(np.random.default_rng(s)).tobytes() for s in range(30)}
    assert len(masks) > 3, "the mask never changes"
    assert (~m.block_mask(np.random.default_rng(1), block=3)).sum() == 9
```

</details>

<details><summary><b>💡 جواب مرجع</b> (بعد از یک تلاش واقعی باز کن)</summary>

```python
"""Step 1: patches and block masks for a mini I-JEPA (8x8 digits -> 4x4 grid of 2x2 patches)."""
from __future__ import annotations

import numpy as np

GRID, PATCH = 4, 2


def patchify(img: np.ndarray) -> np.ndarray:
    """(8, 8) image -> (16, 4): patches row by row, pixels inside a patch row by row."""
    return img.reshape(GRID, PATCH, GRID, PATCH).transpose(0, 2, 1, 3).reshape(GRID * GRID, PATCH * PATCH)


def block_mask(rng: np.random.Generator, block: int = 2) -> np.ndarray:
    """keep mask (16,) bool: True = patch is visible. One random block x block square of the 4x4 patch grid is hidden."""
    keep = np.ones((GRID, GRID), bool)
    r, c = rng.integers(0, GRID - block + 1, size=2)
    keep[r:r + block, c:c + block] = False
    return keep.reshape(-1)
```

</details>

## خروجی مورد انتظار
شش ✓. در اجرای دستی یک رقم ۸×۸ (با `#` و `.`) و شبکه‌ی ۴×۴ ماسک (`X` = پنهان) چاپ می‌شود؛ ۴ خانه‌ی `X` که یک مربع ۲×۲ می‌سازند.

## اگر فرق داشت
| علامت | علت رایج |
|---|---|
| patch ها در هم ریخته‌اند | `transpose` را نگذاشتی یا ترتیب محورها اشتباه است |
| ماسک بیرون شبکه می‌زند | حد بالای `rng.integers` باید `4 - block + 1` باشد |
| ماسک بیش از ۴ patch پنهان کرده | به‌جای مربع، `keep` را اشتباه شاخص‌گذاری کرده‌ای |

## دام‌ها
- ماسک **تصادفی و پراکنده** برای JEPA معمولاً بد است (مدل همسایه را کپی می‌کند)؛ بلوک‌های بزرگ‌تر یادگیری معنایی‌تر می‌دهند. این را بعداً با `block=3` امتحان کن.
- ترتیب patch ها باید با embedding موقعیتی هماهنگ باشد؛ اگر ترتیب را عوض کنی، همه‌چیز بعدی خراب می‌شود.

### 📝 امتحان کوتاه
به هر سؤال با شماره‌ی گزینه جواب بده.

In [ ]:
from learn.notebook import quiz_cell
quiz_cell('p17_jepa_basics', 'step01_patches_masks')

---

# گام ۲: هدف آموزش — خطای پیش‌بینی در فضای embedding و به‌روزرسانی EMA

## هدف این گام
دو قطعه‌ی PyTorch می‌نویسی که قلب I-JEPA‌اند: **loss فقط روی patch های پنهان در فضای embedding** و **به‌روزرسانی EMA** برای encoder معلم (teacher).

## پیش‌نیاز
PyTorch پایه: تانسور، `requires_grad`، `no_grad`. نصب: `pip install -r requirements-jepa.txt`. اگر torch نداری، تست این گام «SKIPPED» می‌شود و می‌توانی از گام‌های دیگر جلو بروی.

## مفهوم
**۱. loss در فضای embedding.** encoder زمینه را می‌بیند و predictor برای هر patch پنهان یک بردار `pred` پیش‌بینی می‌کند. هدف `target` بردار همان patch از encoder معلم (که کل تصویر را دیده) است. خطا فقط روی **patch های پنهان** حساب می‌شود (آن‌هایی که `keep=False`):

```
loss = smooth_l1( pred[~keep], target[~keep] )
```
`smooth_l1` (Huber) مثل MSE برای خطای کوچک و مثل L1 برای خطای بزرگ است: به داده‌ی پرت حساسیت کمتری دارد.

**۲. چرا معلم EMA؟** اگر هدف از همان encoder در حال آموزش بیاید و گرادیان بگیرد، ساده‌ترین راه کاهش loss این است که همه‌ی embedding ها ثابت شوند (collapse). I-JEPA هدف را از یک **کپی کند‌تغییر** encoder می‌گیرد، **بدون گرادیان**:

```
θ_teacher ← m · θ_teacher + (1 − m) · θ_student        (m نزدیک 1، مثل 0.99)
```
به آن میانگین متحرک نمایی (Exponential Moving Average) می‌گویند. معلم همیشه کمی عقب‌تر از دانش‌آموز است، و این «ثبات» از collapse جلوگیری می‌کند (توجیه نظری کامل آن ضعیف است؛ روش بعدی LeJEPA/SIGReg دقیقاً برای جایگزینی این ترفند آمد).

## تمرین کد

1. `masked_latent_loss(pred, target, keep)` : `pred`, `target` به شکل `(B, 16, D)` و `keep` بولی `(B, 16)`. فقط موقعیت‌های `~keep` را بگیر و `F.smooth_l1_loss` را برگردان. اگر هیچ patch ای پنهان نبود، `0` با همان dtype برگردان (نه NaN).
2. `ema_update(teacher, student, m)` : **درجا** (in-place)، بدون گرادیان، برای هر جفت پارامتر `p_t = m * p_t + (1 - m) * p_s`.

### ✍️ کد تو
کد زیر را ببین، TODO ها را کامل کن و سلول را اجرا کن (Shift+Enter). این‌جا می‌نویسی، نه در فایل دیگر.

In [ ]:
"""Step 2: the I-JEPA loss (masked, in embedding space) and the EMA teacher update."""
from __future__ import annotations

import torch
import torch.nn.functional as F


def masked_latent_loss(pred: torch.Tensor, target: torch.Tensor, keep: torch.Tensor) -> torch.Tensor:
    """pred, target: (B, N, D); keep: (B, N) bool, True = visible. The loss only looks at the HIDDEN patches."""
    # TODO 1: hidden = ~keep; if nothing is hidden return pred.sum() * 0.0; otherwise
    #         F.smooth_l1_loss(pred[hidden], target[hidden])   (boolean indexing gives (num_hidden, D))
    raise NotImplementedError("TODO 1: masked_latent_loss")


@torch.no_grad()
def ema_update(teacher: torch.nn.Module, student: torch.nn.Module, m: float = 0.99) -> None:
    """In place: teacher = m * teacher + (1 - m) * student, parameter by parameter."""
    # TODO 2: for p_t, p_s in zip(teacher.parameters(), student.parameters()): p_t.mul_(m).add_(p_s, alpha=1 - m)
    raise NotImplementedError("TODO 2: ema_update")

### ▶ اجرای نمایشی
اول کد را بخوان، بعد اجرا کن. این سلول از توابعی که بالا نوشتی استفاده می‌کند.

In [ ]:
t, s = torch.nn.Linear(2, 2), torch.nn.Linear(2, 2)
before = t.weight.clone()
ema_update(t, s, 0.9)
print("teacher moved by", (t.weight - before).abs().sum().item())

### ✅ بررسی خودکار
تست‌ها را روی چیزی که در سلول‌های بالا نوشتی اجرا می‌کند. ✗ = هنوز نه؛ پیام را بخوان و سلول کد را اصلاح و دوباره اجرا کن.

In [ ]:
from learn.notebook import check_cell
check_cell('p17_jepa_basics', 'step02_latent_loss_ema', globals())

<details><summary><b>🔍 این تست‌ها دقیقاً چه چیزی را می‌سنجند؟</b> (کد تست)</summary>

```python
REQUIRES = ["torch"]


def _t():
    import torch
    return torch


def check_loss_only_on_hidden(m):
    """visible patches do not contribute to the loss"""
    torch = _t()
    pred, target = torch.zeros(1, 4, 2), torch.zeros(1, 4, 2)
    target[0, 0] = 100.0                                   # a huge error on a VISIBLE patch
    keep = torch.tensor([[True, True, False, False]])
    assert float(m.masked_latent_loss(pred, target, keep)) == 0.0, "the visible patch leaked into the loss"


def check_loss_value(m):
    """smooth L1 over hidden patches"""
    torch = _t()
    pred, target = torch.zeros(1, 2, 1), torch.tensor([[[0.5], [3.0]]])
    keep = torch.tensor([[False, False]])
    # smooth_l1: 0.5*0.5^2 = 0.125 for |e|=0.5 ; |e|-0.5 = 2.5 for |e|=3
    assert abs(float(m.masked_latent_loss(pred, target, keep)) - (0.125 + 2.5) / 2) < 1e-6


def check_loss_without_hidden(m):
    """nothing hidden -> zero, not NaN"""
    torch = _t()
    x = torch.randn(2, 3, 4, requires_grad=True)
    out = m.masked_latent_loss(x, torch.randn(2, 3, 4), torch.ones(2, 3, dtype=torch.bool))
    assert float(out) == 0.0 and not torch.isnan(out), out
    out.backward()


def check_loss_has_gradient(m):
    """gradient flows to the prediction"""
    torch = _t()
    pred = torch.randn(2, 4, 3, requires_grad=True)
    m.masked_latent_loss(pred, torch.randn(2, 4, 3), torch.tensor([[True, True, False, False]] * 2)).backward()
    assert pred.grad is not None and pred.grad[:, 2:].abs().sum() > 0 and pred.grad[:, :2].abs().sum() == 0


def check_ema_formula(m):
    """teacher = m*teacher + (1-m)*student"""
    torch = _t()
    t, s = torch.nn.Linear(1, 1, bias=False), torch.nn.Linear(1, 1, bias=False)
    t.weight.data.fill_(1.0)
    s.weight.data.fill_(3.0)
    m.ema_update(t, s, 0.75)
    assert abs(float(t.weight) - (0.75 * 1.0 + 0.25 * 3.0)) < 1e-6, float(t.weight)


def check_ema_no_grad_and_student_untouched(m):
    """the teacher gets no gradient history and the student does not change"""
    torch = _t()
    t, s = torch.nn.Linear(2, 2), torch.nn.Linear(2, 2)
    before = s.weight.clone()
    m.ema_update(t, s, 0.9)
    assert torch.equal(s.weight, before), "student was modified"
    assert t.weight.requires_grad and t.weight.grad_fn is None, "teacher weights must stay plain leaf parameters"
```

</details>

<details><summary><b>💡 جواب مرجع</b> (بعد از یک تلاش واقعی باز کن)</summary>

```python
"""Step 2: the I-JEPA loss (masked, in embedding space) and the EMA teacher update."""
from __future__ import annotations

import torch
import torch.nn.functional as F


def masked_latent_loss(pred: torch.Tensor, target: torch.Tensor, keep: torch.Tensor) -> torch.Tensor:
    """pred, target: (B, N, D); keep: (B, N) bool, True = visible. The loss only looks at the HIDDEN patches."""
    hidden = ~keep
    if not hidden.any():
        return pred.sum() * 0.0
    return F.smooth_l1_loss(pred[hidden], target[hidden])


@torch.no_grad()
def ema_update(teacher: torch.nn.Module, student: torch.nn.Module, m: float = 0.99) -> None:
    """In place: teacher = m * teacher + (1 - m) * student, parameter by parameter."""
    for p_t, p_s in zip(teacher.parameters(), student.parameters()):
        p_t.mul_(m).add_(p_s, alpha=1 - m)
```

</details>

## خروجی مورد انتظار
شش ✓ (یا SKIPPED اگر torch نصب نیست).

## اگر فرق داشت
| علامت | علت رایج |
|---|---|
| loss روی همه‌ی patch ها حساب شد | ماسک `~keep` را اعمال نکرده‌ای؛ patch های دیده‌شده هم هدف نیستند |
| معلم گرادیان گرفت | `ema_update` را در `torch.no_grad()` نگذاشته‌ای |
| پارامترها تغییر نمی‌کنند | `m` را برعکس گذاشتی (معلم باید `m` از خودش نگه دارد) |
| NaN | `smooth_l1_loss` روی تانسور خالی؛ حالت بدون patch پنهان را جدا کن |

## دام‌ها
- هدف را حتماً از معلم بگیر و `detach` کن/زیر `no_grad`؛ اگر گرادیان به هدف برسد، collapse می‌شود.
- `m=0.99` یعنی معلم در هر گام فقط ۱٪ حرکت می‌کند. در آموزش کوتاه (۳۰ epoch) معمولاً ۰٫۹۹ مناسب است؛ آموزش طولانی از ۰٫۹۹۶ تا ۱ برنامه‌ریزی می‌شود.
- loss پایین در JEPA به‌تنهایی **نشانه‌ی خوب بودن نیست**؛ در گام ۳ می‌بینی چرا.

### 📝 امتحان کوتاه
به هر سؤال با شماره‌ی گزینه جواب بده.

In [ ]:
from learn.notebook import quiz_cell
quiz_cell('p17_jepa_basics', 'step02_latent_loss_ema')

---

# گام ۳: تشخیص فروپاشی نمایش (collapse)

## هدف این گام
دو عدد را حساب می‌کنی که collapse را نشان می‌دهند، و تابعی که از آن‌ها تشخیص می‌دهد نمایش `collapsed`، `low_rank` یا `healthy` است. این مهم‌ترین مهارت عملی کار با روش‌های self-supervised است.

## پیش‌نیاز
numpy و کمی جبر خطی (SVD) در حد «مقدارهای تکین نشان می‌دهند داده در چند جهت پخش است».

## مفهوم
**collapse** یعنی encoder همه‌ی ورودی‌ها را (تقریباً) به یک نقطه می‌برد. آن‌وقت پیش‌بینی ساده است (همیشه همان نقطه) و loss صفر می‌شود ولی نمایش بی‌فایده است. loss پایین در این حالت **خبر خوب نیست**. دو نشانه:

1. **میانگین انحراف معیار** هر بُعد بین نمونه‌ها (`mean_std`) نزدیک صفر است: همه‌ی تصویرها یک embedding دارند.
2. **رتبه‌ی مؤثر** (effective rank) کم است: embedding ها فقط در چند جهت پخش‌اند (فروپاشی بُعدی، dimensional collapse) حتی اگر std صفر نباشد.

**رتبه‌ی مؤثر** چطور؟ `z` ماتریس `(N, D)` است. میانگین را کم کن (centering)، مقدارهای تکین `s` را بگیر، آن‌ها را به توزیع تبدیل کن `p = s / sum(s)` و **آنتروپی** بگیر؛ `exp(آنتروپی)` تعداد «جهت‌های مؤثر» است:

```
p = s / s.sum();   effective_rank = exp( -Σ p log p )
```
اگر یک جهت همه‌چیز را توضیح دهد، رتبه ≈ ۱؛ اگر همه‌ی D جهت یکسان استفاده شوند، رتبه ≈ D.

## تمرین کد

1. `collapse_stats(z)` : `{"mean_std": ..., "effective_rank": ..., "dim": D}`. `z` یک آرایه‌ی numpy با شکل `(N, D)`. برای `log(0)` از `np.clip(p, 1e-12, None)` استفاده کن.
2. `diagnose(stats, std_tol=0.01, low_rank_ratio=0.1)` :
   - `"collapsed"` اگر `mean_std < std_tol` یا `effective_rank < 1.5`
   - `"low_rank"` اگر رتبه‌ی مؤثر کمتر از `low_rank_ratio * dim` باشد
   - وگرنه `"healthy"`.

### ✍️ کد تو
کد زیر را ببین، TODO ها را کامل کن و سلول را اجرا کن (Shift+Enter). این‌جا می‌نویسی، نه در فایل دیگر.

In [ ]:
"""Step 3: detect representation collapse from a batch of embeddings."""
from __future__ import annotations

import numpy as np


def collapse_stats(z: np.ndarray) -> dict:
    """z: (N, D) embeddings of N different inputs."""
    # TODO 1: mean_std = mean over the D dimensions of the std ACROSS samples (axis 0)
    # TODO 2: effective_rank: centre z, s = singular values (np.linalg.svd(..., compute_uv=False)), p = s / s.sum()
    #         (if s.sum() == 0 the rank is 0), effective_rank = exp(-sum(p * log(clip(p, 1e-12))))
    raise NotImplementedError("TODO 1-2: collapse_stats")


def diagnose(stats: dict, std_tol: float = 0.01, low_rank_ratio: float = 0.1) -> str:
    # TODO 3: "collapsed" if mean_std < std_tol or effective_rank < 1.5; "low_rank" if effective_rank < low_rank_ratio * dim; else "healthy"
    raise NotImplementedError("TODO 3: diagnose")

### ▶ اجرای نمایشی
اول کد را بخوان، بعد اجرا کن. این سلول از توابعی که بالا نوشتی استفاده می‌کند.

In [ ]:
rng = np.random.default_rng(0)
cases = {"constant": np.ones((200, 16)) + 1e-4 * rng.normal(size=(200, 16)),
         "gaussian": rng.normal(size=(200, 16)),
         "low rank": rng.normal(size=(200, 3)) @ rng.normal(size=(3, 64))}
for name, z in cases.items():
    st = collapse_stats(z)
    print(f"{name:9s} std={st['mean_std']:.4f} rank={st['effective_rank']:.2f}/{st['dim']} -> {diagnose(st)}")

### ✅ بررسی خودکار
تست‌ها را روی چیزی که در سلول‌های بالا نوشتی اجرا می‌کند. ✗ = هنوز نه؛ پیام را بخوان و سلول کد را اصلاح و دوباره اجرا کن.

In [ ]:
from learn.notebook import check_cell
check_cell('p17_jepa_basics', 'step03_collapse', globals())

<details><summary><b>🔍 این تست‌ها دقیقاً چه چیزی را می‌سنجند؟</b> (کد تست)</summary>

```python
import numpy as np

RNG = np.random.default_rng(0)


def check_stats_keys(m):
    """collapse_stats returns mean_std, effective_rank and dim"""
    st = m.collapse_stats(RNG.normal(size=(50, 8)))
    assert set(st) >= {"mean_std", "effective_rank", "dim"} and st["dim"] == 8, st


def check_std_is_across_samples(m):
    """mean_std is measured across samples (axis 0)"""
    z = np.tile(np.arange(5.0), (10, 1))                    # every sample identical, dimensions differ
    assert m.collapse_stats(z)["mean_std"] == 0.0, "std must be taken across samples, not across dimensions"


def check_constant_is_collapsed(m):
    """identical embeddings are collapsed"""
    z = np.ones((100, 16))
    st = m.collapse_stats(z)
    assert st["effective_rank"] == 0.0 or st["effective_rank"] < 1.5, st
    assert m.diagnose(st) == "collapsed", m.diagnose(st)


def check_random_is_healthy(m):
    """isotropic Gaussian embeddings are healthy with rank close to D"""
    st = m.collapse_stats(RNG.normal(size=(500, 16)))
    assert st["effective_rank"] > 12, st
    assert m.diagnose(st) == "healthy", st


def check_low_rank(m):
    """a 3-dimensional cloud inside 64 dimensions is low_rank, not collapsed"""
    z = RNG.normal(size=(400, 3)) @ RNG.normal(size=(3, 64))
    st = m.collapse_stats(z)
    assert 2 < st["effective_rank"] < 4, st
    assert m.diagnose(st) == "low_rank", (m.diagnose(st), st)


def check_one_direction_is_collapsed(m):
    """everything on one line: dimensional collapse"""
    z = RNG.normal(size=(200, 1)) @ RNG.normal(size=(1, 32))
    assert m.diagnose(m.collapse_stats(z)) == "collapsed"
```

</details>

<details><summary><b>💡 جواب مرجع</b> (بعد از یک تلاش واقعی باز کن)</summary>

```python
"""Step 3: detect representation collapse from a batch of embeddings."""
from __future__ import annotations

import numpy as np


def collapse_stats(z: np.ndarray) -> dict:
    """z: (N, D) embeddings of N different inputs."""
    z = np.asarray(z, dtype=float)
    mean_std = float(z.std(axis=0).mean())
    s = np.linalg.svd(z - z.mean(axis=0), compute_uv=False)
    total = s.sum()
    if total == 0:
        rank = 0.0
    else:
        p = np.clip(s / total, 1e-12, None)
        rank = float(np.exp(-(p * np.log(p)).sum()))
    return {"mean_std": mean_std, "effective_rank": rank, "dim": int(z.shape[1])}


def diagnose(stats: dict, std_tol: float = 0.01, low_rank_ratio: float = 0.1) -> str:
    if stats["mean_std"] < std_tol or stats["effective_rank"] < 1.5:
        return "collapsed"
    if stats["effective_rank"] < low_rank_ratio * stats["dim"]:
        return "low_rank"
    return "healthy"
```

</details>

## خروجی مورد انتظار
شش ✓، و در اجرای دستی: ثابت ← `collapsed`، گاوسی ← `healthy`، رتبه‌ی پایین ← `low_rank`.

## اگر فرق داشت
| علامت | علت رایج |
|---|---|
| رتبه‌ی مؤثر برای داده‌ی ثابت NaN شد | تقسیم بر صفر در `p`؛ جمع مقدارهای تکین صفر است (مدیریتش کن) |
| داده‌ی گاوسی `low_rank` شد | centering را فراموش کردی یا std را روی محور اشتباه گرفتی (`axis=0`) |
| `mean_std` برای هر نمونه حساب شد | انحراف معیار باید **بین نمونه‌ها** (محور ۰) باشد |

## دام‌ها
- **رتبه‌ی پایین همیشه collapse نیست.** دنیایی که واقعاً دوبعدی است (مثل نقطه‌ی متحرک در درس world model) رتبه‌ی ۲ تا ۳ دارد و سالم است. collapse یعنی «رتبه و پراکندگی از آنچه داده دارد کمتر است» همراه با loss ≈ ۰.
- `StandardScaler` قبل از linear probe تفاوت‌های ریز را بزرگ می‌کند و collapse را پنهان می‌کند؛ پس probe به‌تنهایی کافی نیست، `mean_std` را هم گزارش کن.
- آستانه‌ها (۰٫۰۱ و ۰٫۱) قرارداد ساده‌اند؛ برای هر پروژه با نمونه‌های سالم خودت تنظیمشان کن.

### 📝 امتحان کوتاه
به هر سؤال با شماره‌ی گزینه جواب بده.

In [ ]:
from learn.notebook import quiz_cell
quiz_cell('p17_jepa_basics', 'step03_collapse')

---

# گام ۴: ارزیابی صادقانه — linear probe، چند seed و «حکم» بر پایه‌ی عدد

## هدف این گام
آزمایش واقعی را روی ارقام اجرا می‌کنی (از `src/jepa_lab`) و دو تابع می‌نویسی که نتیجه را **صادقانه** خلاصه می‌کنند: `summarize` (میانگین و بازه روی چند seed) و `verdict` (آیا روش یادگرفته از baseline ها بهتر است؟ پایدار است؟).

## پیش‌نیاز
گام‌های ۱ تا ۳ مفهوماً. برای اجرای آزمایش PyTorch لازم است؛ خود توابع این گام فقط Python ساده‌اند.

## مفهوم
**linear probe** استاندارد ارزیابی SSL است: encoder را **منجمد** کن، embedding ها را بگیر و یک طبقه‌بند خطی (logistic regression) روی آن آموزش بده. دقت probe یعنی «اطلاعات برچسب چقدر در embedding ساده در دسترس است».

سه ردیف **baseline** همیشه لازم است:
- **پیکسل خام** (بدون encoder): روی ارقام ۸×۸ خیلی قوی است (≈ ۰٫۹۷)، چون این داده کوچک و ساده است.
- **encoder تصادفی** (همان معماری، بدون آموزش): ارزش واقعی «آموزش دیدن» را نشان می‌دهد.
- نسخه‌ی مولد **pixel** (شبیه MAE): رقیب مستقیم JEPA.

**چند seed:** نتیجه‌ی یک seed تصادفی است. در آزمایش ما نسخه‌ی SIGReg یک بار ۰٫۹۳ و بار دیگر ۰٫۶۹ داد! همیشه میانگین، کمینه و بیشینه را گزارش کن.

**حکم** باید قاعده‌مند و قابل‌تست باشد، نه سلیقه‌ای:
- `pixels_win`: آیا پیکسل خام از بهترین روش یادگرفته بهتر یا مساوی است؟
- `beats_random`: کدام روش‌ها میانگین‌شان حداقل `margin` (۰٫۰۲) بالاتر از encoder تصادفی است؟
- `unstable`: کدام روش‌ها بازه‌ی (max − min) بیش از `max_range` (۰٫۱) دارند؟
- `best_learned`: نام بهترین روش یادگرفته (غیر از baseline ها).

## تمرین کد

1. `summarize(rows)` : `rows` لیستی از `{"run": نام, "probe": عدد}` (هر seed یک ردیف). خروجی `{نام: {"mean", "min", "max", "n"}}`.
2. `verdict(summary, margin=0.02, max_range=0.1)` : دیکشنری بالا. نام baseline ها `raw_pixels` و `random_encoder` است؛ بقیه «یادگرفته»اند.

### ✍️ کد تو
کد زیر را ببین، TODO ها را کامل کن و سلول را اجرا کن (Shift+Enter). این‌جا می‌نویسی، نه در فایل دیگر.

In [ ]:
"""Step 4: summarise runs over several seeds and give a rule-based verdict."""
from __future__ import annotations

BASELINES = {"raw_pixels", "random_encoder"}


def summarize(rows: list[dict]) -> dict:
    """rows: [{"run": "ema", "probe": 0.81}, ...] (one row per run and seed) -> {"ema": {"mean", "min", "max", "n"}, ...}"""
    # TODO 1: group the probe values by run name and compute mean, min, max and the count n
    raise NotImplementedError("TODO 1: summarize")


def verdict(summary: dict, margin: float = 0.02, max_range: float = 0.1) -> dict:
    """{"best_learned": name | None, "pixels_win": bool, "beats_random": [names], "unstable": [names]}"""
    # TODO 2: learned = runs that are not in BASELINES; best_learned = the learned run with the highest mean (None if none)
    #         pixels_win = raw_pixels exists and its mean >= the best learned mean (False when raw_pixels is missing)
    #         beats_random = learned runs with mean >= random_encoder mean + margin (empty when random_encoder is missing)
    #         unstable = learned runs with max - min > max_range
    raise NotImplementedError("TODO 2: verdict")

### ▶ اجرای نمایشی
اول کد را بخوان، بعد اجرا کن. این سلول از توابعی که بالا نوشتی استفاده می‌کند.

In [ ]:
import torch

from jepa_lab.ssl import Encoder, linear_probe, load_digits_patches, train

rows = []
for seed in (7, 8):
    data = load_digits_patches(seed)
    torch.manual_seed(seed)
    rows += [{"run": "raw_pixels", "probe": linear_probe(None, data)},
             {"run": "random_encoder", "probe": linear_probe(Encoder(64).eval(), data)}]
    for variant in ("ema", "sigreg", "pixel"):
        rows.append({"run": variant, "probe": linear_probe(train(variant, epochs=15, seed=seed, data=data)["encoder"], data)})
        print("seed", seed, variant, round(rows[-1]["probe"], 3), flush=True)
summ = summarize(rows)
for name, s in sorted(summ.items(), key=lambda kv: -kv[1]["mean"]):
    print(f"{name:15s} {s['mean']:.3f}  [{s['min']:.3f}, {s['max']:.3f}]  n={s['n']}")
print("verdict:", verdict(summ))

### ✅ بررسی خودکار
تست‌ها را روی چیزی که در سلول‌های بالا نوشتی اجرا می‌کند. ✗ = هنوز نه؛ پیام را بخوان و سلول کد را اصلاح و دوباره اجرا کن.

In [ ]:
from learn.notebook import check_cell
check_cell('p17_jepa_basics', 'step04_probe_and_verdict', globals())

<details><summary><b>🔍 این تست‌ها دقیقاً چه چیزی را می‌سنجند؟</b> (کد تست)</summary>

```python
ROWS = [{"run": "raw_pixels", "probe": 0.96}, {"run": "raw_pixels", "probe": 0.97},
        {"run": "random_encoder", "probe": 0.80}, {"run": "random_encoder", "probe": 0.82},
        {"run": "ema", "probe": 0.84}, {"run": "ema", "probe": 0.86},
        {"run": "sigreg", "probe": 0.70}, {"run": "sigreg", "probe": 0.93}]


def check_summarize(m):
    """mean, min, max and n per run"""
    s = m.summarize(ROWS)
    assert set(s) == {"raw_pixels", "random_encoder", "ema", "sigreg"}, set(s)
    assert abs(s["ema"]["mean"] - 0.85) < 1e-9 and s["ema"]["min"] == 0.84 and s["ema"]["max"] == 0.86 and s["ema"]["n"] == 2, s["ema"]


def check_best_learned_ignores_baselines(m):
    """the best learned run is not a baseline"""
    v = m.verdict(m.summarize(ROWS))
    assert v["best_learned"] == "ema", v          # sigreg mean 0.815 < ema 0.85 ; raw_pixels is a baseline


def check_pixels_win(m):
    """raw pixels beat the best learned run here"""
    assert m.verdict(m.summarize(ROWS))["pixels_win"] is True


def check_beats_random(m):
    """only runs at least `margin` above the random encoder"""
    v = m.verdict(m.summarize(ROWS), margin=0.02)
    assert v["beats_random"] == ["ema"], v
    assert m.verdict(m.summarize(ROWS), margin=0.0)["beats_random"] == ["ema", "sigreg"]


def check_unstable(m):
    """a run with a wide range across seeds is flagged"""
    v = m.verdict(m.summarize(ROWS), max_range=0.1)
    assert v["unstable"] == ["sigreg"], v


def check_missing_baselines_do_not_crash(m):
    """no baselines -> pixels_win False, beats_random empty"""
    v = m.verdict(m.summarize([{"run": "ema", "probe": 0.9}]))
    assert v == {"best_learned": "ema", "pixels_win": False, "beats_random": [], "unstable": []}, v
    assert m.verdict({})["best_learned"] is None
```

</details>

<details><summary><b>💡 جواب مرجع</b> (بعد از یک تلاش واقعی باز کن)</summary>

```python
"""Step 4: summarise runs over several seeds and give a rule-based verdict."""
from __future__ import annotations

BASELINES = {"raw_pixels", "random_encoder"}


def summarize(rows: list[dict]) -> dict:
    """rows: [{"run": "ema", "probe": 0.81}, ...] (one row per run and seed) -> {"ema": {"mean", "min", "max", "n"}, ...}"""
    groups: dict[str, list[float]] = {}
    for r in rows:
        groups.setdefault(r["run"], []).append(r["probe"])
    return {k: {"mean": sum(v) / len(v), "min": min(v), "max": max(v), "n": len(v)} for k, v in groups.items()}


def verdict(summary: dict, margin: float = 0.02, max_range: float = 0.1) -> dict:
    """{"best_learned": name | None, "pixels_win": bool, "beats_random": [names], "unstable": [names]}"""
    learned = {k: v for k, v in summary.items() if k not in BASELINES}
    best = max(learned, key=lambda k: learned[k]["mean"]) if learned else None
    pixels_win = "raw_pixels" in summary and best is not None and summary["raw_pixels"]["mean"] >= learned[best]["mean"]
    rand = summary.get("random_encoder")
    beats = [k for k, v in learned.items() if rand and v["mean"] >= rand["mean"] + margin]
    unstable = [k for k, v in learned.items() if v["max"] - v["min"] > max_range]
    return {"best_learned": best, "pixels_win": pixels_win, "beats_random": beats, "unstable": unstable}
```

</details>

## خروجی مورد انتظار
شش ✓ و در اجرای دستی جدولی مثل:
```
raw_pixels      0.97  [0.96, 0.97]
pixel           0.89  ...
ema             0.82  ...
sigreg          0.81  [0.69, 0.93]    <- ناپایدار
random_encoder  0.81
verdict: pixels_win=True ...
```
**تفسیر صحیح:** روی داده‌ی کوچک، پیکسل خام برنده است و روش‌های JEPA فاصله‌ی ناچیزی با encoder تصادفی دارند. این **شکست JEPA نیست**؛ مزیت JEPA در داده‌ی بزرگ و پرجزئیات (تصویر واقعی، ویدیو) و در world model (درس‌های p19–p21) ظاهر می‌شود. گزارش صادقانه همین است.

## اگر فرق داشت
| علامت | علت رایج |
|---|---|
| `KeyError: raw_pixels` | در تست ردیف baseline نیست؛ `verdict` باید با نبودن آن‌ها هم نشکند (اگر نبود، `pixels_win=False`) |
| `unstable` همیشه خالی | بازه را با `max - min` نسنجیده‌ای |
| `best_learned` یک baseline شد | باید baseline ها را از رقابت کنار بگذاری |

## دام‌ها
- «JEPA از encoder تصادفی کمی بهتر است» روی ۲ seed یعنی هیچ؛ حداقل ۳ تا ۵ seed و بازه.
- `margin` را **قبل** از دیدن نتیجه انتخاب کن، وگرنه حکم را به میل خودت تنظیم کرده‌ای.
- کم بودن epoch ها (۱۵ تا ۴۰) همه‌ی روش‌ها را کم‌آموز می‌کند؛ برای نتیجه‌ی جدی‌تر `--epochs 100` را در `scripts/jepa1_ijepa_digits.py` بزن.

## جمع‌بندی p17 و قدم بعد
حالا patch/ماسک، loss و EMA، تشخیص collapse و ارزیابی صادقانه را دست‌به‌کار دیده‌ای. ادامه: [docs/jepa.md](#) مرحله‌ی ۴ (SIGReg، پروژه‌ی p18) و مرحله‌ی ۵ (world model و تصمیم‌گیری با برنامه‌ریزی، پروژه‌ی p19).

### 📝 امتحان کوتاه
به هر سؤال با شماره‌ی گزینه جواب بده.

In [ ]:
from learn.notebook import quiz_cell
quiz_cell('p17_jepa_basics', 'step04_probe_and_verdict')

---
## 🏁 گزارش نهایی درس
نتیجه‌ی تست و امتحان هر گام، و آنچه هنوز مانده.

In [ ]:
from learn.notebook import lesson_report
lesson_report('p17_jepa_basics')